In [2]:
# ── Cell 1: Environment verification ─────────────────────────────────────────
# Project: MV Feeder Protection Study
# Tool:    pandapower + Python
# Region:  South Africa / SADC — Eskom-style 11 kV radial feeder
# Author:  Hillary M
# ─────────────────────────────────────────────────────────────────────────────

import pandapower as pp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Verify pandapower version
print(f"pandapower version: {pp.__version__}")

# Confirm installation is functional
net = pp.create_empty_network()
print("\nEmpty network created successfully:")
print(net)

pandapower version: 3.4.0

Empty network created successfully:
This pandapower network is empty


In [3]:
# ── Cell 2: Network Topology Definition ──────────────────────────────────────
# Eskom-style 11 kV radial feeder — Mpumalanga coalfields context
# Topology: Grid infeed → Transformer → Bus 1 → Line 1→2 → Bus 2 → Line 2→3 → Bus 3
# ─────────────────────────────────────────────────────────────────────────────

net = pp.create_empty_network(name="Mpumalanga 11kV Radial Feeder", f_hz=50, sn_mva=1)

# ── Buses ─────────────────────────────────────────────────────────────────────
bus0 = pp.create_bus(net, vn_kv=132,  name="Bus 0 - Grid Infeed (132kV)")
bus1 = pp.create_bus(net, vn_kv=11,   name="Bus 1 - Zone 1 (Substation 11kV)")
bus2 = pp.create_bus(net, vn_kv=11,   name="Bus 2 - Zone 2 (Mid-Feeder)")
bus3 = pp.create_bus(net, vn_kv=11,   name="Bus 3 - Mine Incomer")

# ── External grid (swing source at Bus 0) ─────────────────────────────────────
# s_sc_max_mva / s_sc_min_mva: max and min short-circuit power of Eskom grid
# rx_max / rx_min: R/X ratio of source impedance
pp.create_ext_grid(net, bus=bus0, vm_pu=1.0, va_degree=0,
                   s_sc_max_mva=250, s_sc_min_mva=120,
                   rx_max=0.1, rx_min=0.1,
                   name="Eskom Grid Infeed")

# ── Transformer: 132/11 kV, 40 MVA ───────────────────────────────────────────
# vk_percent: short-circuit voltage = 10% (leakage impedance)
# vkr_percent: resistive component — X/R = 20, so vkr = vk/20 = 0.5%
# pfe_kw: iron (core) losses; i0_percent: no-load current
pp.create_transformer_from_parameters(
    net,
    hv_bus=bus0, lv_bus=bus1,
    sn_mva=40,
    vn_hv_kv=132, vn_lv_kv=11,
    vkr_percent=0.5,
    vk_percent=10.0,
    pfe_kw=30,
    i0_percent=0.05,
    name="132/11kV 40MVA Transformer"
)

# ── Line 1→2: 9 km Pelican ACSR overhead line ────────────────────────────────
# r_ohm_per_km, x_ohm_per_km: positive-sequence impedance of Pelican conductor
# c_nf_per_km: line capacitance (typical for ACSR OHL)
# max_i_ka: thermal rating = 3.3 MVA / (sqrt(3) × 11 kV) = 0.173 kA
pp.create_line_from_parameters(
    net,
    from_bus=bus1, to_bus=bus2,
    length_km=9,
    r_ohm_per_km=0.306,
    x_ohm_per_km=0.383,
    c_nf_per_km=8.0,
    max_i_ka=0.173,
    name="Line 1-2 (9km Pelican ACSR)"
)

# ── Line 2→3: 6 km Pelican ACSR overhead line ────────────────────────────────
pp.create_line_from_parameters(
    net,
    from_bus=bus2, to_bus=bus3,
    length_km=6,
    r_ohm_per_km=0.306,
    x_ohm_per_km=0.383,
    c_nf_per_km=8.0,
    max_i_ka=0.173,
    name="Line 2-3 (6km Pelican ACSR)"
)

# ── Loads ─────────────────────────────────────────────────────────────────────
# Bus 2: rural/agricultural mix — 400 kW, 200 kVAr
pp.create_load(net, bus=bus2, p_mw=0.4, q_mvar=0.2, name="Rural Load (Bus 2)")

# Bus 3: mine — compressors, winders, crushers — 1.2 MW, 600 kVAr
pp.create_load(net, bus=bus3, p_mw=1.2, q_mvar=0.6, name="Mine Load (Bus 3)")

# ── Circuit breakers (switches) ───────────────────────────────────────────────
# et='l' means line switch; these are the tripping points for relays R1, R2, R3
sw1 = pp.create_switch(net, bus=bus1, element=0, et='l', closed=True, name="CB R1 (Bus 1)")
sw2 = pp.create_switch(net, bus=bus2, element=1, et='l', closed=True, name="CB R2 (Bus 2)")

# ── Verify ────────────────────────────────────────────────────────────────────
print("=" * 55)
print("NETWORK SUMMARY — Mpumalanga 11kV Radial Feeder")
print("=" * 55)
print(f"\nBuses:        {len(net.bus)}")
print(f"Lines:        {len(net.line)}")
print(f"Transformers: {len(net.trafo)}")
print(f"Loads:        {len(net.load)}")
print(f"Switches:     {len(net.switch)}")
print(f"Ext grids:    {len(net.ext_grid)}")

print("\n── Bus table ──")
print(net.bus[['name', 'vn_kv']])

print("\n── Line table ──")
print(net.line[['name', 'length_km', 'r_ohm_per_km', 'x_ohm_per_km', 'max_i_ka']])

print("\n── Load table ──")
print(net.load[['name', 'p_mw', 'q_mvar']])


NETWORK SUMMARY — Mpumalanga 11kV Radial Feeder

Buses:        4
Lines:        2
Transformers: 1
Loads:        2
Switches:     2
Ext grids:    1

── Bus table ──
                               name  vn_kv
0       Bus 0 - Grid Infeed (132kV)  132.0
1  Bus 1 - Zone 1 (Substation 11kV)   11.0
2       Bus 2 - Zone 2 (Mid-Feeder)   11.0
3              Bus 3 - Mine Incomer   11.0

── Line table ──
                          name  length_km  r_ohm_per_km  x_ohm_per_km  \
0  Line 1-2 (9km Pelican ACSR)        9.0         0.306         0.383   
1  Line 2-3 (6km Pelican ACSR)        6.0         0.306         0.383   

   max_i_ka  
0     0.173  
1     0.173  

── Load table ──
                 name  p_mw  q_mvar
0  Rural Load (Bus 2)   0.4     0.2
1   Mine Load (Bus 3)   1.2     0.6
